In [1]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

True
12.1


In [ ]:
# imageTr - training image
# labelTr - training labels
# imageTs - test images

In [ ]:
# 파일 정보
#  파일     의미
# flair   부종 강조
# t1      기본 구조
# t1ce    조영된 종양
# t2      수분 강조
# seg     정답(label)

# 입력 = 4채널(flair, t1, t1ce, t2)
# 출력 = segmentation mask

In [ ]:
import nibabel as nib

img = nib.load("/mnt/c/Users/lhe09/projects/msd_brain/imagesTr/BRATS_001.nii.gz")
data = img.get_fdata()

print(data.shape) # (240, 240, 155, 4)

# 240 * 240 - 이미지 크기
# 155 - depth(slice 수)
# 4 - MRI modality(flair, t1, t1ce, t2)

# PyTorch/3D 모델에서는 반드시 (4, H, W, D) 의 형태여야함.

(240, 240, 155, 4)


In [ ]:
import numpy as np
# 라벨 확인
label = nib.load("/mnt/c/Users/lhe09/projects/msd_brain/labelsTr/BRATS_001.nii.gz").get_fdata()

print(label.shape) # (240, 240, 155)
print(np.unique(label)) # [0. 1. 2. 3.]

(240, 240, 155)
[0. 1. 2. 3.]


In [ ]:
## 만일 gz파일이 4개 라벨이 따로 되어 있으면
# 4개의 modality(flair, t1, t1ce, t2)로 되어 있을경우 합치기
flair = nib.load(...).get_fdata()
t1 = nib.load(...).get_fdata()
t1ce = nib.load(...).get_fdata()
t2 = nib.load(...).get_fdata()

image = np.stack([flair, t1, t1ce, t2], axis=0)

# normalization
image = (image - np.mean(image)) / np.std(image)

# label 처리
label = nib.load(...).get_fdata()
label[label == 4] = 3

In [ ]:
# Segmenatation
# 정의 : 이미지 각 픽셀(또는 voxel)에 "클래스 라벨"을 붙이는것
# 예를 들어 각 voxel -> 0, 1, 2, 3 이면 0-background, 1-edema, 2-non-enhancing tumor, 3-enhancing tumor

# classification 은 이미지 전체 1개 라벨
# detection 은 박스
# segmentation은 픽셀 단위 라벨

## 따라서, segmentation = "dense prediction"

In [ ]:
## Segmentation에서 중요한거!!!
# 1. 모델보다는 데이터 Garbage in -> Garbage out
# 2. Class imbalance 보통 tumor << backgroud가 많으니깐
# -> Dice loss, Focal loss, pos/neg sampling으로 해결
# 3. patch sampling(RandCropByPosNegLabeld) <-이거 없으면 모델은 tumor 못 배움
# 4. nomalization : MRI는 aabsolute intensity 없어서 Normalize가 필수임
# 5. loss funtion : CrossEntorpy 안씀, Dice/DiceFocalLoss가 좋음

In [ ]:
# 나의 궁금증... 왜 registration 없이 segmentation이 가능?
# registration이란? 여러 이미지를 같은 좌표계로 맞추는 것
# 과거에는 MRI -> template에 정렬 -> 분석으로 위치 기반 분석 필요
# 지금은 딥러닝이 위치를 학습함
# 1. CNN특성으로 local pattern학습(tumor모양/texture로 판단)
# 2. 데이터의 다양성으로 여러 위치의 tumor를 학습(위치 invariant)
# 3. augmentation으로 crop/shift/flip을 활용하여 위치 변화 학습
# 따라서 "모델이 직접 spatial understanding을 배움"

# 하지만,
# multi-modal alignment/longitudinal study(시간 비교)/atlas 기반 분석
# 일 경우에는 registration이 필요함!!

In [ ]:
# UNet(segmentation의 표준)
## 구조
# Encoder down(Downsampling path) - 이미지에서 "의미(feature)" 추출(Conv -> ReLU -> Conv -> ReLU -> MaxPool)해상도는 낮추며 진행, 채널은 증가함
# Bottleneck - 가장 압축된 feature(가장 깊은 의미):가장 작은 공간 + 가장 많은 채널(이게 tumork다라는 판단이 거의 끝남)
# Decoder up(Upsampling path) - feature를 다시 원래 해상도로 복원(UpConv -> concat(skip) -> Conv -> Conv)해상도를 다시 올리고 채널을 낮춤
# skip connection(핵심) - encoder의 pooling->위치 정보가 손실됨, 그래서 encoder feature를 Decoder로 직접전달(경계(edge)가 유지되고, 작은 구조 복원되고, segmentation정화도가 올라감)
## 특징
# Downsampling -> context 이해
# Upsampling -> 위치 복원
# skip connection -> 디테일 유지
## 핵심
# "전체 구조 + 디테일 동시에 학습"

In [ ]:
# Dice은 Segmentation에서의 정확도(Accuracy)
# 겹치는 영역이 큰지 보는 지표
# segmentation은 imbalance를 해결하고 overlap직접 최적화하고, boundary에 민감하기 위해 사용
# Dice Loss는 Loss = 1 - Dice : Dice가 높으면 Loss는 낮음


In [ ]:
# Attention UNet(중요한 부분만 보게 만드는 UNet)
# skip connection에 attention 추가
## 효과 : 불필요한 영역은 줄고, tumor 영역은 늘어남
# "UNet + 어디를 볼지 선택"

In [ ]:
# DynUNet(데이터에 맞게 자동으로 구조 조정)
## 특징
# kernel/stride 자동 최적화
# deep supervision 지원
# anisotropic data 대응 - z축은 약하니깐 덜 보자로 해결(그걸 자동으로 kernal/stride를 조정)
## 쓰는 이유
# MRI는 Z축의 resolution이 낮음(이걸 DynUNet이 잘 처리함)
# "UNet의 고급 버전(실전용)"

In [ ]:
# 다른 모델
# 1. DynUNet - patch 기반 최적화, deep surpervision, MSD benchmark강자
# 2. SwinUNETR - Transformer 기반, global context 이해, 작은 tumor 잘 잡음
# 3. UNERT - tansformer + UNet, SwinUNERT 보다 약간 구형
# 4. Res(Residual) 계열 모델
## 입력 -> conv -> conv -> +원래 입력 : gradient 잘흐름, deeper network 가능, feature보존
## 4-1. ResUet, ResUNet++, UNet + ResBlock
# (UNet은 기본차라고 한다면, ResUNet은 튜닝된 스포츠카 느낌)
# 5. nnUNet - 자동 최적화, 의료 segmentation SOTA(구조이해가 어려움, 커스터마이징이 힘듦)
# (UNet은 구조만 정의, 나머지는 다 사람이 설정 / nnUNet은 자동화 시스템으로 모든게 다 자동으로 결정됨)

In [ ]:
# UNet에서 num_res_units를 쓰면 res효과가 있는 건 맞지만 ResUNet은 아님
# UNet 구조는 유지 하되 내부 block만 residual로 변경한 것으로 기본차에 튜닝조금 한거지 스포츠카는 아님
# ResUNet은 encoder 전체가 residual이고 skip connection도 개선되고 더 깊고 복잡한 모델임

In [ ]:
# res, swin
## Res(Residual)의 핵심은 Residual Connection - 학습을 잘 되게 해주는 구조
# 입력 x -> conv -> conv -> + x(skip) : 변화를 배우지, 전체를 새로 배우지 마라
# 학습 깊이가 깊어질수록 gradient vanishing이 일어나 학습이 안되는 걸 residual로 gradient가 shortcut으로 흐르고, 깊어도 학습이 잘됨
# 학습이 덜 터지고 초기화에 덜 민감

## Swin의 핵심은 Swin Transformer - 멀리 보는 능력을 주는 구조
# CNN은 가까운 것만 보지만 Swin은 self-attention으로 멀리 있는 정보까지 한 번에 봄
# global context 이해, long-range dependency(한번에 연결), hierarcical 구조
# 단점은 데이터가 많이 필요, 학습 어려움(lr민감, augmentation 영향이 큼), local detail약함

In [ ]:
## nnUNet은 자동화 시스템이지 모델이 아님
# patch sampling, spacing/resampling, loss, network depth/kernel, postprocessing 이 모든걸 자동으로 맞춰감
# 내부 구조는 Residual UNet + deep supervision으로 이루어짐(encoder-decoder 구조, skip connection, residual block 사용- DynUNet이랑 비슷한 계열)

In [ ]:
# Deep Supervision = "중간 layer들에도 loss를 걸어서 학습시키는 것"
# 일반적인 UNET은 encoder -> decoder -> 최종 output -> loss 여서, 맨 마지막 output의 loss만 걸림
# -> 그럼 gradient가 깊은 layer까지 못가고, early layer는 대충 feature만 뽑는 역할이라서 작은 구조의 학습이 어려움
# 그래서 Deep supervision은 중간 decoder 단계마다 output을 만듦
# decoder stage 1 -> output1 -> loss1
# decoder stage 2 -> output2 -> loss2
# ...
# final output -> loss4
# total loss = loss1 + loss2 + .. + loss4
# 하면 gradient가 훨씬 잘 흐름 각 layer가 직접 supervision을 받으므로 학습 안정성이 높아감
# 그리고 multi-scale학습으로 coarse(큰구조), fine(작은 구조) 모두 동시에 학습됨
# 단점으로는 output이 여러개니 메모리를 더 쓰고, label resize가 필요하여 구현이 귀찮고, inference 는 마지막 output만 사용

In [ ]:
# 3D segmentation을 볼 때 Dice 말고 또 봐야할것!!
# Dice은 예측값과 실제값의 교집합이 어느 정도인지 알려주는 지표
# per-class Dice은 클래스별 Dice
# Hausdorff Distance(HD95)는 예측 경계와 정답 경계 사이의 최대 거리(boundary 정확도)

In [ ]:
# RandCropByPosNegLabeld - foreground vs background 기준, tumor가 포함된 patch를 더 많이 뽑음
#                        - 단점 : tumor 종류(class) 구분 안함, 작은 tumor class 는 여전히 거의 안 나옴
# RandCropByLabelClassed - class 별로 sampling, class 1, 2, 3 각각 고려(모든 tumor cloass를 골고루 보게 함)
#                        - 단점 : 현실 분포랑 다르게 학습됨, false positive 증가, noise 증가